# skforecast Integration Example

This notebook demonstrates how to use timelens with skforecast's `ForecasterRecursiveMultiSeries` (0.21+).

You must pass the same `series` you used in `fit(series=...)` into `from_skforecast(..., series=...)` or `get_training_data(series=...)`, because skforecast's `create_train_X_y` requires it.

**Note:** This notebook requires the `skforecast` optional dependency:
```bash
pip install timelens[skforecast]
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Check if skforecast is available
try:
    from skforecast.recursive import ForecasterRecursiveMultiSeries
    SKFORECAST_AVAILABLE = True
except ImportError:
    SKFORECAST_AVAILABLE = False
    print("skforecast not installed. Install with: pip install timelens[skforecast]")

## 1. Create Sample Data

In [ ]:
np.random.seed(42)
n_periods = 500
dates = pd.date_range('2023-01-01', periods=n_periods, freq='h')

series_data = {}
for i in range(5):
    series_id = f'MT_{i+1:03d}'
    base = np.random.randn() * 10
    trend = np.linspace(0, 5, n_periods)
    seasonal = 5 * np.sin(2 * np.pi * np.arange(n_periods) / 24)
    noise = np.random.randn(n_periods) * 0.5
    series_data[series_id] = base + trend + seasonal + noise

df = pd.DataFrame(series_data, index=dates)
print(f"Data shape: {df.shape}")
df.head()

In [ ]:
# plot the series
df.plot()

## 2. Train ForecasterRecursiveMultiSeries

In [ ]:
if SKFORECAST_AVAILABLE:
    forecaster = ForecasterRecursiveMultiSeries(
        estimator=RandomForestRegressor(
            n_estimators=50,
            max_depth=10,
            random_state=42
        ),
        lags=12
    )
    forecaster.fit(series=df)
    print(f"Forecaster fitted with {forecaster.estimator.n_features_in_} features")
else:
    print("Skipping: skforecast not available")

## 3. Create Adapter and Extract Training Data

In [ ]:
forecaster.summary

In [ ]:
if SKFORECAST_AVAILABLE:
    from timelens.adapters.skforecast import from_skforecast

    # Same `series` as fit(series=...) — required by skforecast.create_train_X_y
    adapter = from_skforecast(forecaster, series=df)
    X, y = adapter.get_training_data()
    
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"Features: {adapter.get_feature_names()}")
    print(f"Series IDs: {adapter.get_series_ids()}")
else:
    print("Skipping: skforecast not available")

## 4. Compute Conditional Importance

In [ ]:
if SKFORECAST_AVAILABLE:
    from timelens import ConditionalPermutationImportance
    from timelens.partitioners import TreePartitioner
    
    partitioner = TreePartitioner(
        random_state=42
    )

    explainer = ConditionalPermutationImportance(
        model=adapter,
        metric='mse',
        strategy='auto',
        partitioner=partitioner,
        n_repeats=3,
        n_jobs=1,
        random_state=42,
    )
    
    result = explainer.compute(X, y)
    print("\nFeature Importance:")
    display(result.to_dataframe())
else:
    print("Skipping: skforecast not available")

## 5. Visualize Results

In [ ]:
if SKFORECAST_AVAILABLE:
    from timelens.visualization import plot_importance_bar
    
    fig, ax = plot_importance_bar(
        result,
        max_features=12,
        title='Conditional Permutation Feature Importance (skforecast)'
    )
else:
    print("Skipping: skforecast not available")

## Summary

This notebook demonstrated the seamless integration between timelens and skforecast:

1. Train a `ForecasterRecursiveMultiSeries` model
2. Create a `SkforecastAdapter` with `from_skforecast(forecaster, series=...)` (same series as `fit`)
3. Extract training data via `adapter.get_training_data()`
4. Use the adapter directly with `ConditionalPermutationImportance`

The adapter handles all the complexity of extracting the training matrix and providing predictions in the format expected by timelens.